In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
torch.manual_seed(0)
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys

torch.manual_seed(0)

In [9]:
# Setup vocabulary
vocab = [
    "[PAD]", "[MASK]", "[UNK]",

    "I", "you", "we", "they",

    # Polysemous words
    "bear",        # animal / tolerate
    "run",         # move / operate
    "bank",        # river / finance
    "charge",      # legal / electrical

    # Nouns
    "river", "road", "field", "court", "battery",
    "money", "load", "power", "side", "shore", "swam",

    # Function words
    "to", "the", "a", "other", "across", "with", "in", "on", "of", "get", "cannot"
]

vocab_size = len(vocab)
print(f'Vocabulary size is {vocab_size}')

Vocabulary size is 33


In [10]:
# Training data
training_data = [
    (["I", "swam", "across", "the", "river", "to", "the", "other", "[MASK]"], "shore"),
    (["they", "went", "to", "the", "bank", "to", "get", "[MASK]"], "money"),
    (["I", "saw", "a", "bear", "in", "the", "[MASK]"], "field"),
    (["I", "cannot", "bear", "the", "[MASK]"], "load"),
    (["they", "run", "across", "the", "[MASK]"], "field"),
    (["the", "battery", "can", "run", "with", "[MASK]"], "power"),
    (["the", "court", "will", "charge", "them", "with", "[MASK]"], "money"),
    (["the", "battery", "has", "a", "charge", "of", "[MASK]"], "power"),
]

In [11]:
sentence = training_data[0][0]
target = training_data[0][1]
idxtoword = {i:j for i,j in enumerate(vocab)}
wordtoidx = {j:i for i,j in enumerate(vocab)}
print(wordtoidx)
def encode(data):
    return torch.tensor([wordtoidx.get(i,wordtoidx["[UNK]"]) for i in data])

{'[PAD]': 0, '[MASK]': 1, '[UNK]': 2, 'I': 3, 'you': 4, 'we': 5, 'they': 6, 'bear': 7, 'run': 8, 'bank': 9, 'charge': 10, 'river': 11, 'road': 12, 'field': 13, 'court': 14, 'battery': 15, 'money': 16, 'load': 17, 'power': 18, 'side': 19, 'shore': 20, 'swam': 21, 'to': 22, 'the': 23, 'a': 24, 'other': 25, 'across': 26, 'with': 27, 'in': 28, 'on': 29, 'of': 30, 'get': 31, 'cannot': 32}


In [12]:
class Transformer(nn.Module):
    def __init__(self,vocab_size,embedingsize):
        super().__init__()

        self.wordEmbedings = nn.Embedding(vocab_size,embedingsize)
        self.q_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.k_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.v_w = nn.Linear(embedingsize,embedingsize,bias=False)

        self.W =  nn.Linear(embedingsize,vocab_size,bias=False)

    def forward(self,sentenceidx,maskidx):
        data = self.wordEmbedings(sentenceidx)
        # print(data)
        Q = self.q_w(data)
        K = self.k_w(data)
        V = self.v_w(data)
        S = F.softmax((Q@K.T)/math.sqrt(Q.size(-1)),dim=-1)

        ouput = S@V

        z = self.W(ouput)

        return z[maskidx]

def lossFun(logets,targetIdx):
    probs = F.softmax(logets,dim=-1)
    # print(probs)
    eps = 1e-09
    return -torch.log(probs[targetIdx]+eps)

In [13]:
vocab_size = len(vocab)
embedingsize = 8
epoch = 500
model = Transformer(vocab_size,embedingsize)
optimizer = torch.optim.Adam(model.parameters(),lr=0.05)
allLoss = []
for i in range(epoch):
    epochloss = 0.0
    random.shuffle(training_data)

    for sentence,target in training_data:
        sentenceidx = encode(sentence)
        # print(sentenceidx)
        maskidx = sentence.index("[MASK]")
        # print(maskidx)
        optimizer.zero_grad()
        targetIdx = wordtoidx[target]
        # print(targetIdx)

        logets = model(sentenceidx,maskidx)
        loss = lossFun(logets,targetIdx)
        loss.backward()
        optimizer.step()
        allLoss.append(loss.item())
        epochloss+=loss.item()
    if i%10==0:
        print(f"epoch {i} : {epochloss:4f}")


epoch 0 : 29.415152
epoch 10 : 0.106830
epoch 20 : 0.010644
epoch 30 : 0.005227
epoch 40 : 0.003177
epoch 50 : 0.002150
epoch 60 : 0.001555
epoch 70 : 0.001176
epoch 80 : 0.000924
epoch 90 : 0.000742
epoch 100 : 0.000609
epoch 110 : 0.000510
epoch 120 : 0.000432
epoch 130 : 0.000371
epoch 140 : 0.000322
epoch 150 : 0.000282
epoch 160 : 0.000248
epoch 170 : 0.000220
epoch 180 : 0.000197
epoch 190 : 0.000176
epoch 200 : 0.000159
epoch 210 : 0.000144
epoch 220 : 0.000130
epoch 230 : 0.000119
epoch 240 : 0.000109
epoch 250 : 0.000100
epoch 260 : 0.000092
epoch 270 : 0.000084
epoch 280 : 0.000078
epoch 290 : 0.000072
epoch 300 : 0.000067
epoch 310 : 0.000062
epoch 320 : 0.000058
epoch 330 : 0.000054
epoch 340 : 0.000050
epoch 350 : 0.000047
epoch 360 : 0.000044
epoch 370 : 0.000041
epoch 380 : 0.000039
epoch 390 : 0.000036
epoch 400 : 0.000034
epoch 410 : 0.000032
epoch 420 : 0.000030
epoch 430 : 0.000028
epoch 440 : 0.000026
epoch 450 : 0.000025
epoch 460 : 0.000023
epoch 470 : 0.000022
ep

In [14]:
no = np.random.randint(0,8)
print(np.random.randint(0,8))
sentence = training_data[no][0]
target = training_data[no][1]

print(sentence)
print(target)


sentenceidx = encode(sentence)
        # print(sentenceidx)
print(sentenceidx)
maskidx = sentence.index("[MASK]")
print(maskidx)

z = model(sentenceidx,maskidx)
probs = F.softmax(z,dim=-1)

print(f"pridiction : {idxtoword[torch.argmax(probs).item()]}")


0
['the', 'battery', 'has', 'a', 'charge', 'of', '[MASK]']
power
tensor([23, 15,  2, 24, 10, 30,  1])
6
pridiction : power
